# The tool loop

A model can write an answer. An agent can also ask our program to do work.

In this notebook, we will:

- describe tools to a model;
- run tools safely;
- build a small text-based tool loop;
- use native tool calls; and
- run the course's finished agent loop.

In [ ]:
import json
import os
from dataclasses import dataclass
from getpass import getpass
from pathlib import Path
from typing import Callable

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
TEMPERATURE = 0
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

## 1. An agent is a loop

The model chooses one of two actions:

1. Ask for a tool.
2. Return the final answer.

If it asks for a tool, our code runs the tool and sends the result back. Then the loop repeats.

## 2. A tool has a public and a private side

The model sees the name, description, and argument shape. It does not see or run the Python function.

In [ ]:
@dataclass
class SimpleTool:
    name: str
    description: str
    parameters: dict
    function: Callable
    risky: bool = False

In [ ]:
from shoplab import world

orders = {order["order_id"]: order for order in world.load_orders()}
customers = {customer["customer_id"]: customer for customer in world.load_customers()}

def get_order(order_id):
    return orders.get(order_id, {"error": "order not found"})

def get_customer(customer_id):
    return customers.get(customer_id, {"error": "customer not found"})

def one_string(name):
    return {
        "type": "object",
        "properties": {name: {"type": "string"}},
        "required": [name],
    }

TOOLS = {
    "get_order": SimpleTool(
        "get_order",
        "Look up one order.",
        one_string("order_id"),
        get_order,
    ),
    "get_customer": SimpleTool(
        "get_customer",
        "Look up one customer.",
        one_string("customer_id"),
        get_customer,
    ),
    "search_policy": SimpleTool(
        "search_policy",
        "Search the shop policies.",
        one_string("query"),
        world.search_policy,
    ),
}

print(list(TOOLS))

## 3. Tool errors are results

A bad tool request should not crash the whole agent. We turn the error into text and send it back to the model.

In [ ]:
def call_tool(name, arguments):
    if name not in TOOLS:
        return json.dumps({"error": f"unknown tool: {name}"})

    try:
        result = TOOLS[name].function(**arguments)
        return json.dumps(result)
    except Exception as error:
        return json.dumps({"error": f"{type(error).__name__}: {error}"})

print(call_tool("get_order", {"order_id": "ORD-7301"})[:120])
print(call_tool("missing_tool", {}))
print(call_tool("get_order", {"wrong_name": "ORD-7301"}))

## 4. Build the loop with a text protocol

First we make the model write a JSON action inside ordinary text. This works, but our prompt and parser must keep the format together.

In [ ]:
from shoplab.llm import complete, parse_json_loose

tool_help = "\n".join(
    f"- {tool.name}: {tool.description}"
    for tool in TOOLS.values()
)

TEXT_INSTRUCTIONS = f"""Help with shop operations.
Reply with one JSON object each turn.
To use a tool: {{"tool": "name", "args": {{...}}}}
To finish: {{"final": "answer"}}
Tools:
{tool_help}"""

def text_agent(task, max_steps=5):
    messages = [
        {"role": "system", "content": TEXT_INSTRUCTIONS},
        {"role": "user", "content": task},
    ]

    for step in range(1, max_steps + 1):
        text = complete(messages, model=MODEL).choices[0].message.content
        action = parse_json_loose(text)
        print("step", step, action)

        if "final" in action:
            return action["final"]

        result = call_tool(action.get("tool", ""), action.get("args", {}))
        messages.append({"role": "assistant", "content": text})
        messages.append({"role": "user", "content": "Tool result: " + result})

    return "stopped: step limit reached"

print(text_agent("What is the total of ORD-7301?"))

## 5. Use native tool calls

Model APIs can carry tool requests in a separate tool_calls field. We no longer need to invent our own text format.

In [ ]:
def api_tools():
    return [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description,
                "parameters": tool.parameters,
            },
        }
        for tool in TOOLS.values()
    ]

messages = [{"role": "user", "content": "What did ORD-7301 cost?"}]
message = complete(messages, model=MODEL, tools=api_tools()).choices[0].message

print("text:", message.content)
print("tool calls:", [call.function.name for call in message.tool_calls or []])

In [ ]:
messages.append(message.model_dump(exclude_none=True))

for request in message.tool_calls or []:
    result = call_tool(
        request.function.name,
        json.loads(request.function.arguments),
    )
    messages.append({
        "role": "tool",
        "tool_call_id": request.id,
        "content": result,
    })

answer = complete(messages, model=MODEL, tools=api_tools())
print(answer.choices[0].message.content)

## 6. Use the course loop

shoplab.loop.run_agent contains the finished native loop.

It stops in three ways:

- finish: the model called the finish tool;
- text: the model returned plain text; or
- max_steps: the loop reached its limit.

In [ ]:
from shoplab.loop import run_agent
from shoplab.tools import Ledger, standard_tools

ledger = Ledger()
desk_tools = standard_tools(ledger)

for tool in desk_tools.values():
    label = "risky" if tool.risky else ""
    print(tool.name, label)

In [ ]:
ticket = next(
    item
    for item in world.load_tickets()["train"]
    if item["ticket_id"] == "TKT-2205"
)

SYSTEM = (
    "You handle returns for Larkspur Outfitters. "
    "Look up the order, customer, and relevant policy. "
    "Then call finish with decision, policy_id, and refund_usd."
)

task = json.dumps({key: value for key, value in ticket.items() if key != "gold"})

def show_step(step, message):
    names = [call.function.name for call in message.tool_calls or []]
    print("step", step, names or [message.content])

result = run_agent(
    task,
    desk_tools,
    model=MODEL,
    system=SYSTEM,
    max_steps=8,
    on_step=show_step,
)

print("stop reason:", result.stop_reason)
print("answer:", result.answer)
print("known:", ticket["gold"])
print("shop actions:", ledger.entries)

## 7. Add simple limits

max_steps stops a loop that keeps going. max_result_chars keeps a large tool result from filling the model's context.

In [ ]:
short_run = run_agent(
    task,
    standard_tools(),
    model=MODEL,
    system=SYSTEM,
    max_steps=1,
    max_result_chars=80,
)

print("stop reason:", short_run.stop_reason)
tool_messages = [message for message in short_run.messages if message.get("role") == "tool"]
if tool_messages:
    print("saved tool text:", repr(tool_messages[0]["content"]))

## What to remember

1. A tool description is data for the model.
2. The Python function stays inside our program.
3. Tool errors should return useful error text.
4. The loop sends tool results back to the model.
5. Native tool calls are safer than a home-made text format.
6. Step and result limits keep the loop bounded.

## Small practice steps

1. Ask text_agent for the tier of CUST-01.
2. Call a missing tool and read the returned error.
3. Change max_steps and watch the stop reason.